# Agent-VScreen 无字元素检测模型（Colab 训练）

**做什么**：训练手机端检测器，找出屏幕上**能点但没有文字**的目标（图标、×、开关、图形按钮），OCR 找不到它们，agent 以前只能猜坐标。

**数据**：RICO 公开数据集（约 6.6 万张真实安卓界面截图 + 无障碍树）。标注由无障碍树自动生成，规则在仓库 `tools/detector/labels.py`，对所有应用一视同仁，不针对任何具体应用。

**模型**（都是一类 `icon` = 可点元素；手机端再用 OCR 区分「有字/无字」）：
| 名字 | 起点 | 用途 | 大小（fp16 存储） |
|---|---|---|---|
| `student` | YOLO11n（COCO 预训练） | **手机默认模型** | 约 5 MB |
| `teacher` | Microsoft OmniParser v2 `icon_detect`（YOLO11m，网页数据训练） | 高精度可选下载、以后给游戏截图做伪标注 | 约 40 MB |

**许可证**：两个模型的权重都继承 Ultralytics YOLO，属于 **AGPL-3.0**，所以单独发布、由 App 按需下载，不打进 App 安装包（App 保持 MIT）。

**已知硬伤**：标注来自无障碍树，游戏和画布界面没有无障碍树，训练数据里几乎没有这类画面。模型在游戏/画布上能不能用，**没有证据**，要在真机上测。RICO 是 2017 年前后的界面，和现在的 App 风格有差距，所以评估时也会单独看你自己采的新截图（如果有）。

## 你要做的
1. 菜单 **代码执行程序 → 更改运行时类型**，选 **GPU**（有 A100 选 A100，其次 L4）。
2. 菜单 **代码执行程序 → 全部运行**。第一格会弹出 Google Drive 授权，点允许。
3. 先用 `MODE = "quick"` 跑一遍（约 30–40 分钟），确认整条链路通了；再把 `MODE` 改成 `"full"`，再 **全部运行**（约 6–7 小时）。
4. 中途断线、运行时被回收都没关系：重新 **全部运行**，会从 Google Drive 里的进度接着训。
5. 跑完后，`我的云端硬盘/vscreen-detector/release/` 里就是要发布的模型、`manifest.json`（含 sha256）和评估结果。把 `manifest.json` 和 `results/summary.md` 的内容发给 Claude 即可。

In [1]:
# ① 设置（只需要改这里）
MODE = "full"          # "quick" = 短训练验证整条链路（约 30–40 分钟）；"full" = 正式训练
STUDENT_HOURS = 0.2 if MODE == "quick" else 3.0   # 手机默认模型的训练时长（小时）
TEACHER_HOURS = 0.2 if MODE == "quick" else 2.5   # OmniParser 微调时长（小时）；设 0 跳过
DATA_FRACTION = 0.15 if MODE == "quick" else 1.0  # 用多少比例的 RICO 训练图
BRANCH = "feat/element-detector"
DRIVE_DIR = "/content/drive/MyDrive/vscreen-detector"

import os, subprocess, sys, json
from google.colab import drive
drive.mount("/content/drive")
RUN_DIR = f"{DRIVE_DIR}/runs-{MODE}"
os.makedirs(RUN_DIR, exist_ok=True); os.makedirs(f"{DRIVE_DIR}/release", exist_ok=True); os.makedirs(f"{DRIVE_DIR}/results", exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/extra", exist_ok=True)
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)

Mounted at /content/drive
name, memory.total [MiB]
NVIDIA A100-SXM4-40GB, 40960 MiB



In [2]:
# ② 安装依赖、取代码
!pip -q install "ultralytics>=8.3.0" onnx
!rm -rf /content/agent-vscreen && git clone -q --depth 1 -b {BRANCH} https://github.com/kilolonion/agent-vscreen /content/agent-vscreen
TOOLS = "/content/agent-vscreen/tools/detector"
print(open(f"{TOOLS}/labels.py").read()[:600])

"""One labelling rule for the textless-element detector, shared by every data source.

A "target" is something a finger can tap that OCR cannot name: an actionable accessibility node (clickable,
long-clickable or checkable), visible, of tap size, whose whole subtree carries no visible text. Actionable nodes
that DO carry text become "ignore" boxes: OCR already finds them, and an icon inside one (a bottom-nav icon above
its label) is neither a positive nor a negative example, so training leaves those pixels out of the loss.

Sources are parsed into the same plain node tree first, then labelled 


In [3]:
# ③ 数据：下载 RICO（6.4 GB，Google 云存储，几分钟）→ 按统一规则生成标注 → 转成 YOLO 格式
import glob
if not os.path.exists("/content/ds/samples.jsonl"):
    if not os.path.isdir("/content/rico/combined"):
        !mkdir -p /content/rico && cd /content/rico && wget -q --show-progress -O - https://storage.googleapis.com/crowdstf-rico-uiuc-4540/rico_dataset_v0.1/unique_uis.tar.gz | tar xz
    !python {TOOLS}/prepare.py rico /content/rico/combined /content/ds --size 448x992
# 你自己采的新截图（可选）：把 PNG/JPG 和同名 uiautomator XML 放进 云端硬盘/vscreen-detector/extra/<任意子目录>/
extra_dirs = []
for d in sorted(glob.glob(f"{DRIVE_DIR}/extra/*/")):
    out = "/content/extra_" + os.path.basename(d.rstrip("/"))
    if not os.path.exists(out + "/samples.jsonl"):
        !python {TOOLS}/prepare.py collected "{d}" "{out}" --size 448x992 --src-tag phone
    extra_dirs.append(out)
# quick 模式只取一部分训练图（验证集不变，评估可比）
src = "/content/ds"
if DATA_FRACTION < 1:
    import random; random.seed(0)
    os.makedirs("/content/ds_sub", exist_ok=True)
    with open("/content/ds/samples.jsonl") as f, open("/content/ds_sub/samples.jsonl", "w") as g:
        for line in f:
            s = json.loads(line)
            if s["split"] == "val" or random.random() < DATA_FRACTION: g.write(line)
    if not os.path.exists("/content/ds_sub/images"): os.symlink("/content/ds/images", "/content/ds_sub/images")
    src = "/content/ds_sub"
!rm -rf /content/yolo && python {TOOLS}/to_yolo.py {src} /content/yolo --extra "{','.join(extra_dirs)}"

-                   100%[===================>]   6.03G  17.9MB/s    in 5m 43s  
5000/66261 read, 4555 kept
10000/66261 read, 9110 kept
15000/66261 read, 13655 kept
20000/66261 read, 18239 kept
25000/66261 read, 22796 kept
30000/66261 read, 27395 kept
35000/66261 read, 31962 kept
40000/66261 read, 36535 kept
45000/66261 read, 41075 kept
50000/66261 read, 45673 kept
55000/66261 read, 50238 kept
60000/66261 read, 54798 kept
65000/66261 read, 59379 kept
done: 60534/66261 screens kept, 186034 textless targets (3.1/screen)
yolo dataset: 57863 train, 2671 val -> /content/yolo/data.yaml


In [4]:
# ④ 对照：不微调的 OmniParser 原版，在同一批验证图上的成绩
import shutil  # Drive 和 /content 是两个文件系统：跨盘只能 move，不能 os.replace
from ultralytics import YOLO
OMNI = "/content/omniparser_icon_detect.pt"
if not os.path.exists(OMNI):
    !wget -q -O {OMNI} https://huggingface.co/microsoft/OmniParser-v2.0/resolve/main/icon_detect/model.pt
if not os.path.exists("/content/omniparser_original_992.onnx"):
    shutil.move(YOLO(OMNI).export(format="onnx", imgsz=[992, 448], opset=17, simplify=False), "/content/omniparser_original_992.onnx")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Ultralytics 8.4.216 🚀 Python-3.13.16 torch-2.14.1+cu130 CPU (Intel Xeon CPU @ 2.20GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino
YOLO11m summary (fused): 125 layers, 20,030,803 parameters, 0 gradients, 73.5 GFLOPs

PyTorch: starting from '/content/omniparser_icon_detect.pt' with input shape (1, 3, 992, 448) BCHW and output shape(s) (1, 5, 9114) (38.7 MB)

ONNX: starting export with onnx 1.23.2 opset 17...
ONNX: export success ✅ 2.2s, saved as '/content/omniparser_icon_detect.onnx' (76.7 MB)

Export complete (3.2s)
Results saved to /content/omniparser_icon_detect.onnx
Predict:         yolo predict task=

In [ ]:
# ⑤ 训练（可中断续训：进度在 云端硬盘/vscreen-detector/runs-<MODE>/）
from ultralytics import YOLO
COMMON = dict(data="/content/yolo/data.yaml", imgsz=992, project=RUN_DIR, exist_ok=True, plots=False,
              fliplr=0.3, mosaic=1.0, close_mosaic=2, hsv_v=0.5, degrees=0, translate=0.1, scale=0.4,
              workers=min(8, os.cpu_count()), cache=False, patience=100)

def train(name, start, hours, batch):
    last = f"{RUN_DIR}/{name}/weights/last.pt"
    if hours <= 0: return None
    if os.path.exists(f"{RUN_DIR}/{name}/weights/best.pt") and os.path.exists(f"{RUN_DIR}/{name}/DONE"):
        print(name, "已经训完"); return f"{RUN_DIR}/{name}/weights/best.pt"
    if os.path.exists(last):
        print(name, "从上次进度继续")
        try:
            YOLO(last).train(resume=True)
        except AssertionError as e:  # 上次其实已经训完（只是没来得及写 DONE）
            print(name, e)
    else:
        YOLO(start).train(name=name, time=hours, batch=batch, **COMMON)
    open(f"{RUN_DIR}/{name}/DONE", "w").close()
    return f"{RUN_DIR}/{name}/weights/best.pt"

STUDENT = train("student", "yolo11n.pt", STUDENT_HOURS, batch=0.7)  # 0.7 = 自动选批大小，占显存 70%
TEACHER = train("teacher", OMNI, TEACHER_HOURS, batch=0.7)
print(STUDENT, TEACHER)

Ultralytics 8.4.216 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=0.7, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=2, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/yolo/data.yaml, degrees=0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.3, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.5, imgsz=992, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=student, nbs=64, nms=None, opset=None, optimize=False,

In [ ]:
# ⑥ 导出手机端模型：ONNX（fp16 存储、fp32 计算，和现有 OCR 同一做法，避开 arm64 fp16 算子的坑）+ sha256
import hashlib, shutil
REL = f"{DRIVE_DIR}/release" if MODE == "full" else f"{DRIVE_DIR}/release-quick"
os.makedirs(REL, exist_ok=True)
exports = {}
for name, pt in (("student", STUDENT), ("teacher", TEACHER)):
    if not pt: continue
    for h, w in ((992, 448), (704, 320)):
        if name == "teacher" and h != 992: continue
        f = YOLO(pt).export(format="onnx", imgsz=[h, w], opset=17, simplify=False)
        dst = f"/content/{name}_{h}x{w}.fp32.onnx"; shutil.move(f, dst)
        out = f"{REL}/vscreen-detector-{name}-{h}x{w}.onnx"
        !python /content/agent-vscreen/tools/ocr-bench/mkstore16.py {dst} /content/tmp16.onnx
        shutil.copy("/content/tmp16.onnx", out)
        exports[f"{name}-{h}x{w}"] = (dst, out)
print(json.dumps(exports, indent=1))

In [ ]:
# ⑦ 评估：用和 App 一样版本的 onnxruntime（1.21.1）读模型，在验证集上问 agent 关心的问题：
#    hit_rate = 无字目标里，有检测框中心落在目标内的比例（找到且能点中）
#    lands_on_tappable = 检测框中心落在可点元素上的比例（点下去不落空）
!pip -q uninstall -y onnxruntime-gpu onnxruntime; pip -q install onnxruntime==1.21.1
RES = f"{DRIVE_DIR}/results"
runs = {"omniparser-original-992x448": "/content/omniparser_original_992.onnx"}
runs.update({k: v[1] for k, v in exports.items()})
rows = []
for k, path in runs.items():
    out = f"{RES}/{MODE}-{k}.json"
    !python {TOOLS}/yolo_eval.py "{path}" {src} --limit 2000 --out "{out}"
    m = json.load(open(out)); b = m["by_threshold"][str(m["best_threshold"])]
    rows.append((k, m["mb"], m["ms_1thread_here"], m["best_threshold"], b["hit_rate"], b["lands_on_tappable"], b["dets_per_screen"]))
    for d in extra_dirs:  # 你自己的新截图单独评一次（看 RICO 旧界面和现在 App 的差距）
        !python {TOOLS}/yolo_eval.py "{path}" {d} --split val --limit 0 --out "{RES}/{MODE}-{k}-{os.path.basename(d)}.json"
md_ = ["| 模型 | MB | 单线程 ms（Colab CPU） | 阈值 | 无字目标命中率 | 落在可点元素上 | 每屏框数 |", "|---|---|---|---|---|---|---|"]
md_ += [f"| {r[0]} | {r[1]} | {r[2]} | {r[3]} | {r[4]:.3f} | {r[5]:.3f} | {r[6]} |" for r in rows]
open(f"{RES}/summary-{MODE}.md", "w").write("\n".join(md_)); print("\n".join(md_))

In [ ]:
# ⑧ 发布清单 manifest.json（App 下载时校验 sha256；阈值取验证集上最好的那个）
def sha(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""): h.update(b)
    return h.hexdigest()
models = []
for k, (_, out) in exports.items():
    m = json.load(open(f"{RES}/{MODE}-{k}.json"))
    name, size = k.split("-", 1); h, w = map(int, size.split("x"))
    models.append({"id": k, "file": os.path.basename(out), "sha256": sha(out), "bytes": os.path.getsize(out),
                   "input": {"h": h, "w": w, "layout": "NCHW", "range": "0..1 RGB", "letterbox": "top-left, pad 114/255"},
                   "output": "[1,5,N]: cx,cy,w,h (input px), score; NMS on device",
                   "threshold": m["best_threshold"], "val": m["by_threshold"][str(m["best_threshold"])],
                   "license": "AGPL-3.0", "default": k == "student-992x448"})
manifest = {"format": 1, "kind": "vscreen-detector", "classes": ["icon"], "mode": MODE, "models": models,
            "notes": "Labels from accessibility trees (RICO). No evidence yet on games/canvas screens."}
json.dump(manifest, open(f"{REL}/manifest.json", "w"), indent=1, ensure_ascii=False)
open(f"{REL}/LICENSE-MODELS.txt", "w").write("These model weights are derived from Ultralytics YOLO11 / Microsoft OmniParser icon_detect and are licensed AGPL-3.0. They are distributed separately from the Agent-VScreen app (MIT).\n")
print(json.dumps(manifest, indent=1, ensure_ascii=False)[:3000])
print("\n发布文件在：", REL)

## 跑完之后
- `quick` 跑通后：把 ① 里的 `MODE` 改成 `"full"`，再 **全部运行**。数据会重新下载处理（约 15 分钟），训练接着放在 `runs-full/` 里。
- `full` 跑完后：把 `release/manifest.json` 和 `results/summary-full.md` 的内容贴给 Claude；模型文件留在 Drive 里，Claude 会告诉你怎么上传到 GitHub Release / ModelScope。
- Colab 过期前没训完也没关系：`runs-full/student/weights/best.pt` 是目前最好的一版，可以直接从 ⑥ 开始导出（先运行 ①②③④，再跳到 ⑥）。